# Bloco 4 — Notebook 03: Estresse de Cauda (`CVaR 99%`), Controle de Drawdown `CPPI` e Modo Fantasma `CUSUM`
Este notebook avalia a proteção patrimonial do **Bloco 4** (`src/risk/position_sizer.py`, `src/risk/shadow_manager.py`, `src/risk/profit_splitter.py`) sobre a estratégia homologada `K9.3`:
- Comparação entre `CPPI_DRAWDOWN_CONTROL` (redução de 50% do lote ao atingir 4% de drawdown), `CUSUM_SHADOW_MODE` (4 vitórias virtuais com $\mathbb{E}[R] > 0$) e `DISABLED`.
- Medição do *Max Drawdown*, *Calmar Ratio*, $\text{CVaR}_{99\%}$ e acúmulo no Cofre de Lucros (`MAIN_FUNDING`) com regra de *High-Water Mark*.


In [ ]:
# Bootstrap automático no Google Colab
import os, subprocess, sys
from pathlib import Path
REPO_URL = 'https://github.com/pac-coins/quant-crypto-chameleon-radar-engine.git'
REPO_DIR = Path('quant-crypto-chameleon-radar-engine')
if 'google.colab' in sys.modules:
    if not Path('config/strategies.yaml').exists():
        if not REPO_DIR.exists():
            subprocess.run(['git', 'clone', REPO_URL], check=True)
        os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pydantic', 'pydantic-settings', 'PyYAML', 'pandas', 'requests', 'cryptography'], check=True)
if '.' not in sys.path:
    sys.path.insert(0, '.')
for _mod in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[_mod]
os.environ.setdefault('MODE', 'SHADOW_CHAMELEON')
os.environ.setdefault('TELEGRAM_BOT_TOKEN', 'colab_backtest_token_placeholder')
os.environ.setdefault('TELEGRAM_CHAT_ID', '0')


In [ ]:
from decimal import Decimal
import numpy as np
from src.core.config import load_config
from src.risk.shadow_manager import ShadowManager
from src.risk.profit_splitter import ProfitSplitter

cfg = load_config()

def simulate_equity_curve_with_cppi_and_splitter(r_series, initial_usdt=Decimal('300.00'), cppi_enabled=True, splitter_enabled=False):
    prot_cfg = cfg.strategies.drawdown_and_shadow_protection.model_copy(
        update={'protection_mode': 'CPPI_DRAWDOWN_CONTROL' if cppi_enabled else 'DISABLED'}
    )
    split_cfg = cfg.strategies.profit_splitter.model_copy(update={'enabled': splitter_enabled})
    sm = ShadowManager(prot_cfg, cfg.strategies.portfolio)
    ps = ProfitSplitter(split_cfg)
    equity = Decimal(str(initial_usdt))
    max_dd = Decimal('0')
    total_transferred = Decimal('0')
    for r_val in r_series:
        status = sm.evaluate_defense(current_equity_usdt=equity)
        if status.drawdown_pct > max_dd:
            max_dd = status.drawdown_pct
        eff_risk = Decimal('0.010') * status.risk_multiplier
        eq_before = equity
        pnl = eq_before * eff_risk * Decimal(str(round(float(r_val), 6)))
        equity = max(Decimal('0'), eq_before + pnl)
        split_res = ps.on_trade_closed(
            equity_before_usdt=eq_before,
            equity_after_usdt=equity,
            cppi_drawdown_active=status.cppi_active,
        )
        if split_res.transferred_to_funding_usdt > Decimal('0'):
            equity -= split_res.transferred_to_funding_usdt
            total_transferred += split_res.transferred_to_funding_usdt
    return {
        'final_tactical_usdt': equity,
        'transferred_funding_usdt': total_transferred,
        'pending_bucket_usdt': ps.pending_hodl_bucket_usdt,
        'max_drawdown_pct': float(max_dd),
    }
